# Setup

In [ ]:
%run common.py

In [ ]:
us_crossreference_path = '../../legal-networks-data/us/4_crossreference_graph/seqitems'
de_crossreference_path = '../../legal-networks-data/de/4_crossreference_graph/seqitems'

In [ ]:
de_graph_files = sorted(list_dir(de_crossreference_path, 'gpickle.gz'))
de_graphs = [nx.read_gpickle(f'{de_crossreference_path}/{gf}') for gf in de_graph_files]

In [ ]:
us_graph_files = sorted(list_dir(us_crossreference_path, 'gpickle.gz'))
us_graphs = [nx.read_gpickle(f'{us_crossreference_path}/{gf}') for gf in us_graph_files]

In [ ]:
years = list(range(1994,1994+min(len(de_graphs),len(us_graphs)))) # usually our xs

# Korpusgröße in Tokens

In [ ]:
de_tokens_n_abs = np.array([
    sum([ndata['tokens_n'] for n, ndata in G.nodes(data=True) if ndata['level'] == 0])
    for G in de_graphs]
)
de_tokens_n_rel = abs_to_rel(de_tokens_n_abs)

us_tokens_n_abs = np.array([
    sum([ndata['tokens_n'] for n, ndata in G.nodes(data=True) if ndata['level'] == 0])
    for G in us_graphs]
)
us_tokens_n_rel = abs_to_rel(us_tokens_n_abs)

In [ ]:
last = 0
for year, x in zip(years, us_tokens_n_abs):
    if not x > last:
        assert False, f'{year} is lower than previous'
    last = x

In [ ]:
last = 0
for year, x in zip(years, de_tokens_n_abs):
    if not x > last and year != 2007:
        assert False, f'{year} is lower than previous'
    last = x

In [ ]:
df_tokens = pd.DataFrame({
    'USA': us_tokens_n_rel,
    'USA (absolut)': us_tokens_n_abs,
    'Deutschland': de_tokens_n_rel,
    'Deutschland (absolut)': de_tokens_n_abs,
    'Jahr': years
}) 
os.makedirs(data_pickles_path, exist_ok=True)
df_tokens.to_pickle(f'{data_pickles_path}/makro_tokens.pickle')

In [ ]:
df_tokens = pd.read_pickle(f'{data_pickles_path}/makro_tokens.pickle')
chart1 = alt.Chart(df_tokens).transform_fold(
    ['USA', 'Deutschland'],
).mark_line().encode(
    alt.X('Jahr:O'),
    alt.Y('value:Q', scale=alt.Scale(zero=False, bins=alt.ScaleBinParams(step=0.1)), title=f'Anzahl der Tokens relativ zu {years[0]}'),
    alt.Color(
        'key:N', 
        legend=None,
    ),
    alt.Tooltip(df_tokens.columns.to_list()),
)
chart2 = chart1.mark_point(filled=True, opacity=1).encode(
    alt.Color(
        'key:N', 
        legend=alt.Legend(title='Land', legendX=10, legendY=2, orient='none', fillColor="#fff", padding=10),
    ),
    alt.Shape('key:N', scale=alt.Scale(range=['circle','triangle-up'])),
)
chart = (chart1 + chart2).resolve_scale(color="independent", shape="independent")
save_chart(chart, 'makro_tokens_relative')

In [ ]:
chart_graycolor = chart.encode(
    alt.Color(
        'key:N', 
        legend=alt.Legend(title='Land', legendX=10, legendY=2, orient='none', fillColor="#fff", padding=10),
        scale=alt.Scale(range=['#888', '#000'])
    ),
)
save_chart(chart_graycolor, 'makro_tokens_relative_graycolor')

In [ ]:
us_tokens_n_abs[-2]/de_tokens_n_abs[-2]

In [ ]:
diss_data('makro_erstes_jahr', str(years[0]))
diss_data('makro_letztes_jahr', str(years[-1]))
diss_data('makro_token_us_wachstumsfaktor', de_num_format(f'{us_tokens_n_rel[-1]:.2f}'))
diss_data('makro_token_us_erstes_jahr', format_mio(us_tokens_n_abs[0]))
diss_data('makro_token_us_letztes_jahr', format_mio(us_tokens_n_abs[-1]))
diss_data('makro_token_de_wachstumsfaktor', de_num_format(f'{de_tokens_n_rel[-1]:.2f}'))
diss_data('makro_token_de_erstes_jahr', format_mio(de_tokens_n_abs[0]))
diss_data('makro_token_de_letztes_jahr', format_mio(de_tokens_n_abs[-1]))

## Warum wächst DE von 2015 bis 2018?

### Quantifizierung des Unterschieds

In [ ]:
diss_data(
    'makro_token_de_differenz_absolut_durchschnitt', 
    format_mio(
        (de_tokens_n_abs[-1]-de_tokens_n_abs[0])/(len(de_tokens_n_abs)-1)
    )
)
diss_data(
    'makro_token_us_differenz_absolut_durchschnitt', 
    format_mio(
        (us_tokens_n_abs[-1]-us_tokens_n_abs[0])/(len(us_tokens_n_abs)-1)
    )
)

In [ ]:
df_tokens_indexed = df_tokens.set_index('Jahr')
de_diff_rel = (
    df_tokens_indexed['Deutschland'][2018] -
    df_tokens_indexed['Deutschland'][2015]
)

us_diff_rel = (
    df_tokens_indexed['USA'][2018] -
    df_tokens_indexed['USA'][2015]
)


de_diff_abs = (
    df_tokens_indexed['Deutschland (absolut)'][2018] -
    df_tokens_indexed['Deutschland (absolut)'][2015]
)

us_diff_abs = (
    df_tokens_indexed['USA (absolut)'][2018] -
    df_tokens_indexed['USA (absolut)'][2015]
)

diss_data('makro_token_de_differenz_relativ_prozentpunkte_2015_2018_durchschnitt', 
          de_num_format(f'{de_diff_rel/(2018-2015)*100:.2f}'))
diss_data('makro_token_us_differenz_relativ_prozentpunkte_2015_2018_durchschnitt', 
          de_num_format(f'{us_diff_rel/(2018-2015)*100:.2f}'))
diss_data('makro_token_de_differenz_absolut_2015_2018', format_mio(de_diff_abs))
diss_data('makro_token_us_differenz_absolut_2015_2018', format_mio(us_diff_abs))

### Welche Gesetze verursachen den Zuwachs in DE?

In [ ]:
def get_tokens_for_abks(G, abk_dict):
    law_nodes = [(n, data) for n, data in G.nodes(data=True) if data.get('type') == 'document']
    for n, data in law_nodes:
        abk_dict[n.split('_')[1]] = data.get('law_name') 
    return {
        n.split('_')[1]: data['tokens_n'] 
        for n, data in law_nodes
    }

def token_delta(G0, G1):
    abk_dict = dict()

    law_tokens0 = get_tokens_for_abks(G0, abk_dict)
    law_tokens1 = get_tokens_for_abks(G1, abk_dict)
    all_abks = set(law_tokens0.keys()).union(law_tokens1.keys())
    law_tokens_delta = dict()
    
    for abk in all_abks:
        law_tokens_delta[abk] = (
            law_tokens1.get(abk, 0) -
            law_tokens0.get(abk, 0)
        )
        
    df = pd.DataFrame(law_tokens_delta.items(), columns=['citekey', 'token_diff']).sort_values('token_diff')
    df['law_name'] = [abk_dict[citekey] for citekey in df['citekey']]
        
    return df

In [ ]:
G2015 = de_graphs[2015-1994]
G2018 = de_graphs[2018-1994]
df = token_delta(G2015, G2018)
laws = df[-6:]['law_name'].to_list()

# shorten long titles a bit
if not '%%'.join(laws) == '%%'.join([
    'Gesetz für den Ausbau erneuerbarer Energien',
    'Gesetz über das Aufspüren von Gewinnen aus schweren Straftaten',
    'Gesetz zum Schutz vor der schädlichen Wirkung ionisierender Strahlung',
    'Sozialgesetzbuch (SGB) Fünftes Buch (V) - Gesetzliche Krankenversicherung - (Artikel 1 des Gesetzes v. 20. Dezember 1988, BGBl. I S. 2477)',
    'Sozialgesetzbuch Neuntes Buch – Rehabilitation und Teilhabe von Menschen mit Behinderungen – (Artikel 1 des Gesetzes v. 23. Dezember 2016, BGBl. I S. 3234)',
    'Gesetz über die Beaufsichtigung der Versicherungsunternehmen'
]):
    print(json.dumps(laws, indent=4, ensure_ascii=False))
    assert False
laws = [
    'Gesetz für den Ausbau erneuerbarer Energien',
    'Gesetz über das Aufspüren von Gewinnen aus schweren Straftaten',
    'Gesetz zum Schutz vor der schädlichen Wirkung ionisierender Strahlung',
    'Sozialgesetzbuch Fünftes Buch',
    'Sozialgesetzbuch Neuntes Buch',
    'Gesetz über die Beaufsichtigung der Versicherungsunternehmen'
]
laws.reverse()
diss_data('makro_token_de_2015_2018_wachsende_gesetze', format_list(laws))
df
# laws

# Korpusgröße in ungeordnete Elemente

In [ ]:
def remove_empty(G: nx.Graph):
    return nx.subgraph(G, [n for n in G.nodes if G.nodes[n]['chars_n'] > 0]).copy()

us_law_graphs = [
    remove_empty(nx.read_gpickle(f'../../legal-networks-data/us/10_preprocessed_graph/{y}_0-0_1-0_-1.gpickle.gz'))
    for y in years
]

de_law_graphs = [
    remove_empty(nx.read_gpickle(f'../../legal-networks-data/de/10_preprocessed_graph/{y}-01-01_0-0_1-0_-1.gpickle.gz'))
    for y in years
]

In [ ]:
de_documents_n_abs = np.array([get_node_count_for_type(G, ['document']) for G in de_graphs])
us_documents_n_abs = np.array([get_node_count_for_type(G, ['document']) for G in us_graphs])
de_buecher_n_abs = np.array([len(G.nodes) for G in de_law_graphs])
us_chapter_n_abs = np.array([len(G.nodes) for G in us_law_graphs])

de_documents_n_rel = abs_to_rel(de_documents_n_abs)
us_documents_n_rel = abs_to_rel(us_documents_n_abs)
de_buecher_n_rel = abs_to_rel(de_buecher_n_abs)
us_chapter_n_rel = abs_to_rel(us_chapter_n_abs)

In [ ]:
diss_data('makro_document_us_erstes_jahr', de_num_format(f'{us_documents_n_abs[0]:,}'))
diss_data('makro_document_us_letztes_jahr', de_num_format(f'{us_documents_n_abs[-1]:,}'))
diss_data('makro_document_de_erstes_jahr', de_num_format(f'{de_documents_n_abs[0]:,}'))
diss_data('makro_document_de_letztes_jahr', de_num_format(f'{de_documents_n_abs[-1]:,}'))
diss_data('makro_document_de_max_jahr', de_num_format(f'{max(de_documents_n_abs):,}'))

diss_data('makro_item_us_chapter_erstes_jahr', de_num_format(f'{us_chapter_n_abs[0]:,}'))
diss_data('makro_item_us_chapter_letztes_jahr', de_num_format(f'{us_chapter_n_abs[-1]:,}'))
diss_data('makro_item_us_chapter_wachstumsfaktor', de_num_format(f'{us_chapter_n_rel[-1]:.2f}'))

diss_data('makro_item_de_buecher_erstes_jahr', de_num_format(f'{de_buecher_n_abs[0]:,}'))
diss_data('makro_item_de_buecher_letztes_jahr', de_num_format(f'{de_buecher_n_abs[-1]:,}'))

In [ ]:
df_documents = pd.DataFrame({
    'USA': us_documents_n_rel,
    'USA (absolut)': us_documents_n_abs,
    'Deutschland': de_documents_n_rel,
    'Deutschland (absolut)': de_documents_n_abs,
    'Deutschland Gesetze/Bücher': de_documents_n_rel,
    'Deutschland Gesetze/Bücher (absolut)': de_documents_n_abs,
    'USA Chapter': us_chapter_n_rel,
    'USA Chapter (absolut)': us_chapter_n_abs,
    'Jahr': years
})

In [ ]:
documents_chart1 = alt.Chart(df_documents).transform_fold(
    ['Deutschland', 'USA', 'USA Chapter'],
).mark_line().encode(
    alt.X('Jahr:O'),
    alt.Y('value:Q', 
          scale=alt.Scale(clamp=True, zero=False), 
          title=['Anzahl der ungeordneten', f'Elemente relativ zu {years[0]}']
         ),
    alt.Color(
        'key:N', 
        legend=None,
        scale=alt.Scale(range=['#4c78a8', '#f58518', '#F9B97E'])
    ),
    alt.Tooltip(df_documents.columns.to_list()),
)
documents_chart2 = documents_chart1.mark_point(filled=True, opacity=1).encode(
    alt.Shape('key:N', scale=alt.Scale(range=['circle','triangle-up','triangle-down'])),
    alt.Color(
        'key:N', 
        legend=alt.Legend(
            title='Land', legendX=10, legendY=2, orient='none', fillColor="#fff", padding=10
        ),
        scale=alt.Scale(range=['#4c78a8', '#f58518', '#F9B97E'])
    ),
)
documents_chart = (documents_chart1 + documents_chart2).resolve_scale(color="independent", shape="independent")
save_chart(documents_chart, 'makro_documents_relative')

In [ ]:
documents_chart1_graycolor = documents_chart1.encode(
    alt.Color(
        'key:N', 
        legend=None,
        scale=alt.Scale(range=['#888', '#000', '#AAA']),
    ),
)
documents_chart2_graycolor = documents_chart2.encode(
    alt.Color(
        'key:N', 
        legend=alt.Legend(
            title='Land', legendX=10, legendY=2, orient='none', fillColor="#fff", padding=10
        ),
        scale=alt.Scale(range=['#888', '#000', '#AAA']),
    ),
)
documents_chart_graycolor = (documents_chart1_graycolor + documents_chart2_graycolor).resolve_scale(color="independent", shape="independent")
save_chart(documents_chart_graycolor, 'makro_documents_relative_graycolor')

## Rückgang Deutschland 2007

In [ ]:
df_documents_indexed = df_documents.set_index('Jahr')
diss_data('makro_de_documents_2007_rueckgang', (
    df_documents_indexed['Deutschland (absolut)'][2006] - 
    df_documents_indexed['Deutschland (absolut)'][2007]
))

# Korpusgröße in indexierten Elementen

In [ ]:
de_seqitems_n_abs = np.array([get_node_count_for_type(G, ['seqitem']) for G in de_graphs])
us_seqitems_n_abs = np.array([get_node_count_for_type(G, ['seqitem']) for G in us_graphs])
de_seqitems_n_rel = abs_to_rel(de_seqitems_n_abs)
us_seqitems_n_rel = abs_to_rel(us_seqitems_n_abs)

In [ ]:
last = 0
for year, x in zip(years, us_seqitems_n_rel):
    if x < last:
        print(f'{year} is lower than previous')
    last = x
us_seqitems_n_rel

In [ ]:
diss_data('makro_seqitem_us_wachstumsfaktor', de_num_format(f'{us_seqitems_n_rel[-1]:,.2f}'))
diss_data('makro_seqitem_us_erstes_jahr', de_num_format(f'{us_seqitems_n_abs[0]:,}'))
diss_data('makro_seqitem_us_letztes_jahr', de_num_format(f'{us_seqitems_n_abs[-1]:,}'))
diss_data('makro_seqitem_de_wachstumsfaktor', de_num_format(f'{de_seqitems_n_rel[-1]:.2f}'))
diss_data('makro_seqitem_de_erstes_jahr', de_num_format(f'{de_seqitems_n_abs[0]:,}'))
diss_data('makro_seqitem_de_letztes_jahr', de_num_format(f'{de_seqitems_n_abs[-1]:,}'))

In [ ]:
df_seqitems = pd.DataFrame({
    'USA': us_seqitems_n_rel,
    'USA (absolut)': us_seqitems_n_abs,
    'Deutschland': de_seqitems_n_rel,
    'Deutschland (absolut)': de_seqitems_n_abs,
    'Jahr': years
})
df_seqitems.to_pickle(f'{data_pickles_path}/makro_seqitems.pickle')

In [ ]:
seqitems_chart1 = alt.Chart(df_seqitems).transform_fold(
    ['USA', 'Deutschland'],
).mark_line().encode(
    alt.X('Jahr:O'),
    alt.Y('value:Q', scale=alt.Scale(zero=False, domain=(1, 1.30), bins=alt.ScaleBinParams(step=0.05)), title=['Anzahl der indexierten', f'Elemente relativ zu {years[0]}']),
    alt.Color('key:N', legend=None),
    alt.Tooltip(df_seqitems.columns.to_list()),
)
seqitems_chart2 = seqitems_chart1.mark_point(filled=True, opacity=1).encode(  
    alt.Shape('key:N', scale=alt.Scale(range=['circle','triangle-up'])),
    alt.Color('key:N', legend=alt.Legend(title='Land', legendX=10, legendY=2, orient='none', fillColor="#fff", padding=10)),
)
seqitems_chart = (seqitems_chart1 + seqitems_chart2).resolve_scale(color="independent", shape="independent")
save_chart(seqitems_chart, 'makro_seqitems_relative')

In [ ]:
seqitems_chart_graycolor = seqitems_chart.encode(
    alt.Color('key:N', legend=alt.Legend(title='Land', legendX=10, legendY=2, orient='none', fillColor="#fff", padding=10), scale=alt.Scale(range=['#888', '#000']))
)
save_chart(seqitems_chart_graycolor, 'makro_seqitems_relative_graycolor')

In [ ]:
df_seqitems

## Warum ist DE 2006 ein lokales Maximum?

In [ ]:
def count_laws(G, abk_dict, node_type):
    law_counter = Counter()

    for n, data in G.nodes(data=True):
        if data['level'] >= 0 and data['type'] == node_type:
            law_counter.update([
                data['key'].split('_')[1]
            ])
            abk_dict[data['key'].split('_')[1]] = data.get('law_name')
    
    return law_counter

def law_name_delta(G0, G1, node_type):
    abk_dict = dict()

    counter0 = count_laws(G0, abk_dict, node_type)
    counter1 = count_laws(G1, abk_dict, node_type)

    counter_removed = counter0 - counter1
    counter_added = counter1 - counter0
    counter_removed_negative = [(abk, -cnt) for abk, cnt in counter_removed.most_common()]

    counter_merged = list(counter_added.most_common())
    counter_merged.extend(counter_removed_negative)

    df = pd.DataFrame(counter_merged, columns=['citekey', 'count'])
    df['law_name'] = [abk_dict[citekey] for citekey in df['citekey']]
    df = df.sort_values('count', ascending=True)
    return df

In [ ]:
G2006 = de_graphs[2006-1994]
G2007 = de_graphs[2007-1994]
df = law_name_delta(G2006, G2007, 'seqitem')

added_laws_sum = df[df['count'] > 0]['count'].sum()
diss_data('makro_de_seqitems_2006_hinzugefuegt_summe', de_num_format(f'{added_laws_sum:,}'))

removed_laws_sum = -df[df['count'] < 0]['count'].sum()
diss_data('makro_de_seqitems_2006_entfernt_summe', de_num_format(f'{removed_laws_sum:,}'))

removed_laws = format_list(df['law_name'][:5].to_list())
diss_data('makro_de_seqitems_2006_entfernt_gesetzesnamen', removed_laws)
print('\n*',"\n* ".join(df['law_name'][:20].to_list()))

In [ ]:
diss_data('makro_de_seqitems_2006_entfernt_top5_quote', de_num_format(f"{df[:5]['count'].sum()/df['count'].sum()*100:.2f}"))

# Korpusgröße in geordneten und ungeordneten Elementen

In [ ]:
de_items_n_abs = np.array([get_node_count_for_type(G, ['item', 'document']) for G in de_graphs])
us_items_n_abs = np.array([get_node_count_for_type(G, ['item', 'document']) for G in us_graphs])
de_items_n_rel = abs_to_rel(de_items_n_abs)
us_items_n_rel = abs_to_rel(us_items_n_abs)

In [ ]:
diss_data('makro_item_us_wachstumsfaktor', de_num_format(f'{us_items_n_rel[-1]:,.2f}'))
diss_data('makro_item_us_erstes_jahr', de_num_format(f'{us_items_n_abs[0]:,}'))
diss_data('makro_item_us_letztes_jahr', de_num_format(f'{us_items_n_abs[-1]:,}'))
diss_data('makro_item_de_wachstumsfaktor', de_num_format(f'{de_items_n_rel[-1]:.2f}'))
diss_data('makro_item_de_erstes_jahr', de_num_format(f'{de_items_n_abs[0]:,}'))
diss_data('makro_item_de_letztes_jahr', de_num_format(f'{de_items_n_abs[-1]:,}'))

In [ ]:
df_items = pd.DataFrame({
    'USA': us_items_n_rel,
    'USA (absolut)': us_items_n_abs,
    'Deutschland': de_items_n_rel,
    'Deutschland (absolut)': de_items_n_abs,
    'Jahr': years
})
df_items.to_pickle(f'{data_pickles_path}/makro_items.pickle')

In [ ]:
items_chart1 = alt.Chart(df_items).transform_fold(
    ['USA', 'Deutschland'],
).mark_line().encode(
    alt.X('Jahr:O'),
    alt.Y('value:Q', 
          scale=alt.Scale(zero=False, bins=alt.ScaleBinParams(step=0.05)), 
          title=['Anzahl der ungeordneten und', f' geordneten Elemente relativ zu {years[0]}']
         ),
    alt.Color('key:N', legend=None),
    alt.Tooltip(df_items.columns.to_list()),
)
items_chart2 = items_chart1.mark_point(filled=True, opacity=1).encode(
    alt.Color('key:N', legend=alt.Legend(
        title='Land', legendX=10, legendY=2, orient='none', fillColor="#fff", padding=10
    )),
    alt.Shape('key:N', scale=alt.Scale(range=['circle','triangle-up'])),    
)
items_chart = (items_chart1 + items_chart2).resolve_scale(color="independent", shape="independent")
save_chart(items_chart, 'makro_items_relative')

In [ ]:
items_chart_graycolor = items_chart.encode(
    alt.Color('key:N', legend=alt.Legend(
        title='Land', legendX=10, legendY=2, orient='none', fillColor="#fff", padding=10), scale=alt.Scale(range=['#888', '#000']))
)
save_chart(items_chart_graycolor, 'makro_items_relative_graycolor')

## Verläuft im wesentlichen gleich wie indexierte Elemente

In [ ]:
seqitems_chart

In [ ]:
items_chart

In [ ]:
de_items_n_rel

## Rückgang DE 1997

In [ ]:
G1996 = de_graphs[1996-1994]
G1997 = de_graphs[1997-1994]
df = law_name_delta(G1996, G1997, 'item')

added_laws_sum = df[df['count'] > 0]['count'].sum()
diss_data('makro_de_items_1997_hinzugefuegt_summe', de_num_format(f'{added_laws_sum:,}'))

removed_laws_sum = -df[df['count'] < 0]['count'].sum()
diss_data('makro_de_items_1997_entfernt_summe', de_num_format(f'{removed_laws_sum:,}'))

In [ ]:
diss_data('makro_de_items_1997_rueckgang', removed_laws_sum - added_laws_sum)

In [ ]:
removed_law = df['law_name'].iloc[0]
assert removed_law == 'Reichsversicherungsordnung'

diss_data('makro_de_items_1997_rvo_entfernt', -df['count'].iloc[0])
          
added_law = df['law_name'].iloc[-1]
assert added_law.startswith('Siebtes Buch Sozialgesetzbuch')

diss_data('makro_de_items_1997_SGB-7_hinzugefuegt', df['count'].iloc[-1])

# Token pro indexiertem Elementen

## Durchschnitt

In [ ]:
seletected_cols = ['USA (absolut)', 'Deutschland (absolut)']
df_tokens_seqitems = (
    df_tokens_indexed[seletected_cols] / 
    df_seqitems.set_index('Jahr')[seletected_cols]
)
df_tokens_seqitems = df_tokens_seqitems.reset_index()
df_tokens_seqitems.columns = ["Jahr", "USA", 'Deutschland']

In [ ]:
df_tokens_seqitems['USA'].iloc[0]

In [ ]:
diss_data('makro_toke_seqitem_rel_us_erstes_jahr', de_num_format(f"{df_tokens_seqitems['USA'].iloc[0]:,.2f}"))
diss_data('makro_toke_seqitem_rel_us_letztes_jahr', de_num_format(f"{df_tokens_seqitems['USA'].iloc[-1]:,.2f}"))
diss_data('makro_toke_seqitem_rel_us_wachstumsfaktor', 
          de_num_format(f"{df_tokens_seqitems['USA'].iloc[-1]/df_tokens_seqitems['USA'].iloc[0]:,.2f}")
         )
diss_data('makro_toke_seqitem_rel_de_erstes_jahr', de_num_format(f"{df_tokens_seqitems['Deutschland'].iloc[0]:,.2f}"))
diss_data('makro_toke_seqitem_rel_de_letztes_jahr', de_num_format(f"{df_tokens_seqitems['Deutschland'].iloc[-1]:,.2f}"))
diss_data('makro_toke_seqitem_rel_de_wachstumsfaktor', 
          de_num_format(f"{df_tokens_seqitems['Deutschland'].iloc[-1]/df_tokens_seqitems['Deutschland'].iloc[0]:,.2f}")
         )

In [ ]:
tokens_seqitems_chart1 = alt.Chart(df_tokens_seqitems).transform_fold(
    ['USA', 'Deutschland'],
).mark_line().encode(
    alt.X('Jahr:O'),
    alt.Y('value:Q', title=['Anzahl der Tokens /', 'Anzahl der indexierten Elemente'], scale=alt.Scale(bins=alt.ScaleBinParams(step=50))),
    alt.Color('key:N', legend=None),
)
tokens_seqitems_chart2 = tokens_seqitems_chart1.mark_point(filled=True, opacity=1).encode(
    alt.Color('key:N', legend=alt.Legend(title='Land', legendX=310, legendY=20, orient='none', fillColor="#fff", padding=10)),
    alt.Shape('key:N', scale=alt.Scale(range=['circle','triangle-up'])),
)
tokens_seqitems_chart = (tokens_seqitems_chart1 + tokens_seqitems_chart2).resolve_scale(color="independent", shape="independent")
save_chart(tokens_seqitems_chart, 'makro_tokens_seqitems_mean')

In [ ]:
tokens_seqitems_chart_graycolor = tokens_seqitems_chart.encode(
    alt.Color('key:N', legend=alt.Legend(title='Land', legendX=310, legendY=20, orient='none', fillColor="#fff", padding=10), scale=alt.Scale(range=['#888', '#000'])),
)
save_chart(tokens_seqitems_chart_graycolor, 'makro_tokens_seqitems_mean_graycolor')

## Histogramm

### Helper

In [ ]:
def get_seqitem_tokens(G):
    return [
        data['tokens_n'] 
        for n, data in G.nodes(data=True) 
        if data['level'] >= 0 and data['type'] == 'seqitem'
    ]

def hist_tokens_seqitems(graphs, years, color):
#     if single_year:
#         graphs = [graphs[years.index(single_year)]]
#         years = [single_year]
        
    bins = np.arange(0, 1000+1, 10)
    
    # Calculate
    binned_all = pd.DataFrame()
    for idx, year in enumerate(years):
        G = graphs[idx]
        seqitems = get_seqitem_tokens(G)
        df = binned_df(
            seqitems, 
            bins=bins,
            year=year
        )
        binned_all = binned_all.append(df)
    
    # Create chart
    chart = alt.Chart(binned_all).mark_bar(
        binSpacing=0, 
        color=color
    ).encode(
        alt.X('bin_min', 
            bin='binned', 
            title="Token",
            axis=alt.Axis(tickCount=len(bins)/5, labelAngle=-90),
        ),
        alt.X2('bin_max'),
        alt.Y('x',
            title="Indexierten Elemente",
            scale=alt.Scale(domain=(0, 3500)),
        ),
        alt.Facet('Jahr:O')
    ).properties(
        width=200, height=100
    )
#     if single_year:
#         chart = chart.properties(
#             width=500, height=100
#         )
#     else:
#         chart = chart.properties(
#             width=200, height=100, columns=5
#         )
        
    return chart

### All years

In [ ]:
chart = hist_tokens_seqitems(de_graphs, years, color='#1f77b4')
save_chart(chart, 'makro_tokens_seqitems_hist_de')
None

In [ ]:
chart = hist_tokens_seqitems(us_graphs, years, color='#ff7f0e')
save_chart(chart, 'makro_tokens_seqitems_hist_us')
None

In [ ]:
seqitems_countries = {
    country: [get_seqitem_tokens(G) for G in graphs]
    for country, graphs 
    in {"Deutschland": de_graphs, 'USA': us_graphs}.items()
}
bins = np.arange(0, 500+1, 10)

binned_all = pd.DataFrame()
for country, seqitems in seqitems_countries.items():
    for idx, year in enumerate(years):
        df = binned_df(seqitems[idx], bins, year)
        df['Land'] = country
        binned_all = binned_all.append(df)
binned_all['Land und Jahr'] = [f'{l} {j}' for l, j in zip(binned_all['Land'], binned_all['Jahr'])]

# Plot 1994 in front of 2019
binned_all = binned_all.sort_values(['Land', 'Jahr'], ascending=False)

# Create chart
charts = []
for land, domain_max in [("Deutschland", 3500), ("USA", 2100)]:
    chart = alt.Chart(binned_all).transform_filter(
        ((alt.datum.Jahr == years[0]) | (alt.datum.Jahr == years[-1])) & (alt.datum.Land == land)
    ).mark_line(interpolate="step-after", size=2).encode(
        alt.X('bin_min', 
            title=("Token" if land == "USA" else None),
            axis=(alt.Axis(tickCount=len(bins)/5, grid=False) if land == "USA" else None),
        ),
        alt.Y('x',
            title="Indexierten Elemente",
            scale=alt.Scale(domain=(0, domain_max), bins=alt.ScaleBinParams(step=500))
        ),
        alt.Row("Land:N", title=None),
        alt.StrokeDash("Jahr:N", scale=alt.Scale(range=[[1, 0], [1, 1]])),
        alt.Color('Jahr:N', scale=alt.Scale(scheme='set1'), legend=alt.Legend(legendX=340, legendY=5, orient='none', fillColor="#fff", padding=10)),
    ).properties(height=domain_max/23)
    charts.append(chart)
chart = alt.vconcat(*charts).configure_concat(spacing=0)
save_chart(chart, 'makro_tokens_seqitems_hist_start_end')

In [ ]:
charts_graycolor = [
    chart.encode(
        alt.Color('Jahr:N', scale=alt.Scale(range=['#000', '#666']), legend=alt.Legend(legendX=340, legendY=5, orient='none', fillColor="#fff", padding=10))
    )
    for chart in charts
]
chart_graycolor = alt.vconcat(*charts_graycolor).configure_concat(spacing=0)
save_chart(chart_graycolor, 'makro_tokens_seqitems_hist_start_end_graycolor')

In [ ]:
binned_all

In [ ]:
std_seqitems_countries = {
    country: [np.std(data) for data in years]
    for country, years 
    in seqitems_countries.items()
}
df_std_seqitems_countries = pd.DataFrame(std_seqitems_countries)
df_std_seqitems_countries['Jahr'] = years

In [ ]:
diss_data('makro_token_seqitem_std_us_erstes_jahr', de_num_format(f"{df_std_seqitems_countries['USA'].iloc[0]:,.2f}"))
diss_data('makro_token_seqitem_std_us_letztes_jahr', de_num_format(f"{df_std_seqitems_countries['USA'].iloc[-1]:,.2f}"))
diss_data('makro_token_seqitem_std_us_rel_letztes_jahr', de_num_format(
    f"{df_std_seqitems_countries['USA'].iloc[-1]/df_std_seqitems_countries['USA'].iloc[0]:,.2f}"
))
diss_data('makro_token_seqitem_std_de_erstes_jahr', de_num_format(f"{df_std_seqitems_countries['Deutschland'].iloc[0]:,.2f}"))
diss_data('makro_token_seqitem_std_de_letztes_jahr', de_num_format(f"{df_std_seqitems_countries['Deutschland'].iloc[-1]:,.2f}"))
diss_data('makro_token_seqitem_std_de_rel_letztes_jahr', de_num_format(
    f"{df_std_seqitems_countries['Deutschland'].iloc[-1]/df_std_seqitems_countries['Deutschland'].iloc[0]:,.2f}"
))

In [ ]:
tokens_seqitems_std_chart1 = alt.Chart(df_std_seqitems_countries).transform_fold(
    ['USA', 'Deutschland'],
).mark_line().encode(
    alt.X('Jahr:O'),
    alt.Y('value:Q', title=f'Standardabweichung', scale=alt.Scale(bins=alt.ScaleBinParams(step=100))),
    alt.Color('key:N', legend=None),
).properties(height=180)
tokens_seqitems_std_chart2 = tokens_seqitems_std_chart1.mark_point(filled=True, opacity=1).encode(
    alt.Shape('key:N', scale=alt.Scale(range=['circle','triangle-up'])),
    alt.Color('key:N', legend=alt.Legend(title='Land', legendX=310, legendY=45, orient='none', fillColor="#fff", padding=10)),
)
tokens_seqitems_std_chart = (tokens_seqitems_std_chart1 + tokens_seqitems_std_chart2).resolve_scale(color="independent", shape="independent")
save_chart(tokens_seqitems_std_chart, 'makro_tokens_seqitems_std')

In [ ]:
tokens_seqitems_std_chart_graycolor = tokens_seqitems_std_chart.encode(
    alt.Color('key:N', scale=alt.Scale(range=['#888', '#000']), legend=alt.Legend(title='Land', legendX=310, legendY=45, orient='none', fillColor="#fff", padding=10)),
)
save_chart(tokens_seqitems_std_chart_graycolor, 'makro_tokens_seqitems_std_graycolor')

# Gliederungstiefe

## Verteilung

In [ ]:
def get_seqitem_levels(G):
    return [data['level'] for n, data in G.nodes(data=True) if data['level'] >= 0 and data['type'] == 'seqitem']

def hist_seqitem_levels(graphs, years, color, single_year=None, hide_x_title=False):    
    # Calculate
    seqitems = [get_seqitem_levels(G) for G in graphs]
    bins = np.arange(0, 9, 1)
    
    level_max = max([max(x) for x in seqitems])
    assert level_max <= len(bins)
    
    binned_all = pd.DataFrame()
    for idx, year in enumerate(years):
        df = binned_df(seqitems[idx], bins, year)
        binned_all = binned_all.append(df)
    
    # Create chart
    chart = alt.Chart(binned_all).mark_bar(binSpacing=0, color=color).encode(
        alt.X('bin_max:O', 
            title=None if hide_x_title else "Gliederungstiefe",
            axis=alt.Axis(tickCount=len(bins), labelAngle=-90),
            scale=alt.Scale(zero=True),
        ),
        alt.Y('x',
            title="Indexierten Elemente",
            scale=alt.Scale(domain=(0,25000)),
        ),
        alt.Facet('Jahr:O')
#     ).properties(
#         columns=5
    )

    return chart

In [ ]:
chart = hist_seqitem_levels(de_graphs, years, color='#1f77b4')
save_chart(chart, 'makro_seqitems_levels_hist_de')
None

In [ ]:
chart = hist_seqitem_levels(us_graphs, years, color='#ff7f0e')
save_chart(chart, 'makro_seqitems_levels_hist_us')
None

In [ ]:
# Compare first and last of both countries

# Calculate
seqitem_levels_countries = {
    country: [get_seqitem_levels(G) for G in graphs]
    for country, graphs 
    in {"Deutschland": de_graphs, 'USA': us_graphs}.items()
}
bins = np.arange(0, 9, 1)

binned_all = pd.DataFrame()
for country, seqitems in seqitem_levels_countries.items():
    for idx, year in enumerate(years):
        df = binned_df(seqitems[idx], bins, year)
        df['Land'] = country
        binned_all = binned_all.append(df)


# Plot 1994 in front of 2019
binned_all = binned_all.sort_values(['Land', 'Jahr'], ascending=False)

# Create chart
chart = alt.Chart(binned_all).transform_filter(
    (alt.datum.Jahr == years[0]) | (alt.datum.Jahr == years[-1])
).mark_bar(opacity=.6).encode(
    alt.X('bin_max:O', 
        title="Gliederungstiefe",
        axis=alt.Axis(tickCount=len(bins), labelAngle=-90),
    ),
    alt.Y('x',
        title="Indexierte Elemente",
#         scale=alt.Scale(domain=(0,25000)),
        stack=None,
    ),
    row='Land:O',
    color=alt.Color('Jahr:N', scale=alt.Scale(scheme='set1')),
).properties(
    width=300,
    height=125
)
save_chart(chart, 'makro_seqitems_levels_hist_start_end')

## Durchschnitt

In [ ]:
seqitem_levels_avg_countries = {
    country: [pd.Series(levels).mean() for levels in levels_years]
    for country, levels_years
    in seqitem_levels_countries.items()
}

df_seqitem_levels = pd.DataFrame(seqitem_levels_avg_countries)
df_seqitem_levels['Jahr'] = years

In [ ]:
seqitem_levels_chart = alt.Chart(df_seqitem_levels).transform_fold(
    ['USA', 'Deutschland'],
).mark_line(point=True).encode(
    alt.X('Jahr:O'),
    alt.Y('value:Q', 
          title=f'Gliederungstiefe',
          scale=alt.Scale(zero=False)
         ),
    alt.Color(
        'key:N', 
        legend=alt.Legend(
            title='Land', 
            legendX=400, 
            legendY=220, 
            fillColor="#fff",
            padding=10
        )
    ),
).properties(height=300)
seqitem_levels_chart